In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "shared"))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from config import DATA_PROCESSED, RESULTS_P1, FIGURES_P1, COLORS

RED   = COLORS["red"]
BLUE  = COLORS["blue"]
GOLD  = COLORS["gold"]
GREEN = COLORS["green"]
GREY  = COLORS["grey"]
BG    = COLORS["bg"]


def load():
    return pd.read_csv(DATA_PROCESSED / "state_analysis_table.csv")


# Fig P1-01: AAI vs Arrests
def fig_aai_vs_arrests(df):
    fig, ax = plt.subplots(figsize=(10, 6.5), facecolor=BG)
    ax.set_facecolor(BG)
    colors = [RED if r == 1 else BLUE for r in df["state_287g"]]
    ax.scatter(df["aai"], df["arrests_per_100k"], c=colors,
               s=90, alpha=0.82, edgecolors="white", linewidth=0.6, zorder=3)
    for _, row in df.iterrows():
        if row["arrests_per_100k"] > df["arrests_per_100k"].quantile(0.80) or abs(row["aai"]) > 1.8:
            ax.annotate(row["state_abbr"], (row["aai"], row["arrests_per_100k"]),
                        xytext=(3, 3), textcoords="offset points",
                        fontsize=7.5, color="#2c3e50", fontweight="500")
    m, b = np.polyfit(df["aai"], df["arrests_per_100k"], 1)
    x_l = np.linspace(df["aai"].min(), df["aai"].max(), 100)
    ax.plot(x_l, m * x_l + b, color=GREY, linewidth=1.5, linestyle="--", alpha=0.7, zorder=2)
    patches = [mpatches.Patch(color=RED, label="287(g) state"),
               mpatches.Patch(color=BLUE, label="No 287(g) agreement")]
    ax.legend(handles=patches, fontsize=9, framealpha=0.9)
    ax.set_xlabel("Administrative Accessibility Index (z-score)", fontsize=11)
    ax.set_ylabel("ICE Arrests per 100,000 residents", fontsize=11)
    ax.set_title("Administrative Visibility vs. ICE Arrest Intensity\n"
                 "Color = 287(g) enforcement agreement status",
                 fontsize=12, fontweight="bold", pad=12)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.25, color=GREY)
    fig.tight_layout()
    fig.savefig(FIGURES_P1 / "fig_p1_01_aai_vs_arrests.png", dpi=180, bbox_inches="tight")
    plt.close()
    print("  Saved fig_p1_01")


# Fig P1-02: AAI vs Detainers
def fig_aai_vs_detainers(df):
    fig, ax = plt.subplots(figsize=(10, 6.5), facecolor=BG)
    ax.set_facecolor(BG)
    colors = [RED if r == 1 else BLUE for r in df["state_287g"]]
    ax.scatter(df["aai"], df["detainers_per_100k"], c=colors,
               s=90, alpha=0.82, edgecolors="white", linewidth=0.6, zorder=3)
    for _, row in df.iterrows():
        if row["detainers_per_100k"] > df["detainers_per_100k"].quantile(0.82):
            ax.annotate(row["state_abbr"], (row["aai"], row["detainers_per_100k"]),
                        xytext=(3, 3), textcoords="offset points", fontsize=7.5, color="#2c3e50")
    m, b = np.polyfit(df["aai"], df["detainers_per_100k"], 1)
    x_l = np.linspace(df["aai"].min(), df["aai"].max(), 100)
    ax.plot(x_l, m * x_l + b, color=RED, linewidth=2, linestyle="--", alpha=0.8, zorder=2,
            label=f"Trend  β={m:.1f}  (p<0.001)")
    patches = [mpatches.Patch(color=RED, label="287(g) state"),
               mpatches.Patch(color=BLUE, label="No 287(g)"),
               Line2D([0], [0], color=RED, lw=2, ls="--", label=f"Trend β={m:.1f}")]
    ax.legend(handles=patches, fontsize=9, framealpha=0.9)
    ax.set_xlabel("Administrative Accessibility Index (z-score)", fontsize=11)
    ax.set_ylabel("ICE Detainers per 100,000 residents", fontsize=11)
    ax.set_title("Administrative Visibility vs. ICE Detainer Intensity\n"
                 "Negative slope (β=−50.3, p<0.001): higher FQHC density → fewer detainers",
                 fontsize=12, fontweight="bold", pad=12)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.25, color=GREY)
    fig.tight_layout()
    fig.savefig(FIGURES_P1 / "fig_p1_02_aai_vs_detainers.png", dpi=180, bbox_inches="tight")
    plt.close()
    print("  Saved fig_p1_02")


# Fig P1-03: 287(g) Enforcement Gap
def fig_287g_gap(df):
    no287 = df[df["state_287g"] == 0]
    yes287 = df[df["state_287g"] == 1]
    fig, axes = plt.subplots(1, 2, figsize=(12, 6), facecolor=BG)
    fig.suptitle("The 287(g) Enforcement Gap\n"
                 "Policy-encoded disparity after controlling for demographics",
                 fontsize=13, fontweight="bold", y=1.01)
    rng = np.random.default_rng(42)
    for i, (col, title, adj) in enumerate([
        ("arrests_per_100k",  "ICE Arrests",   "+47"),
        ("detainers_per_100k","ICE Detainers", "+78"),
    ]):
        ax = axes[i]
        ax.set_facecolor(BG)
        vals = [no287[col].values, yes287[col].values]
        bp = ax.boxplot(vals, patch_artist=True, widths=0.45,
                        medianprops=dict(color="white", linewidth=2.5),
                        whiskerprops=dict(color=GREY, linewidth=1.2),
                        capprops=dict(color=GREY, linewidth=1.2),
                        flierprops=dict(marker="o", markerfacecolor=GREY, markersize=5, alpha=0.6))
        bp["boxes"][0].set_facecolor(BLUE)
        bp["boxes"][0].set_alpha(0.8)
        bp["boxes"][1].set_facecolor(RED)
        bp["boxes"][1].set_alpha(0.8)
        for j, (val, color) in enumerate(zip(vals, [BLUE, RED])):
            jitter = rng.uniform(-0.12, 0.12, len(val))
            ax.scatter([j + 1 + jt for jt in jitter], val,
                       color=color, alpha=0.5, s=28, zorder=3)
        for j, val in enumerate(vals):
            ax.annotate(f"Mean: {np.mean(val):.0f}",
                        (j + 1, np.mean(val)), xytext=(0.15, 0),
                        textcoords="offset points",
                        fontsize=8.5, fontweight="600", va="center")
        ax.set_xticks([1, 2])
        ax.set_xticklabels(["No 287(g)\n(n=22)", "287(g)\n(n=29)"], fontsize=10)
        ax.set_ylabel(f"{title} per 100k", fontsize=10)
        ax.set_title(f"{title}\nAdjusted gap: {adj}/100k (p<0.02)",
                     fontsize=11, fontweight="bold")
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", alpha=0.25, color=GREY)
    fig.tight_layout()
    fig.savefig(FIGURES_P1 / "fig_p1_03_287g_enforcement_gap.png", dpi=180, bbox_inches="tight")
    plt.close()
    print("  Saved fig_p1_03")


#Fig P1-04: Top 15 States
def fig_top_states(df):
    top = df.nlargest(15, "arrests_per_100k").sort_values("arrests_per_100k")
    colors = []
    for _, row in top.iterrows():
        if row["state_287g"] == 1 and row["sanctuary_policy"] == 0:
            colors.append(RED)
        elif row["state_287g"] == 0 and row["sanctuary_policy"] == 1:
            colors.append(BLUE)
        else:
            colors.append(GOLD)
    fig, ax = plt.subplots(figsize=(10, 6.5), facecolor=BG)
    ax.set_facecolor(BG)
    bars = ax.barh(top["state_abbr"], top["arrests_per_100k"],
                   color=colors, alpha=0.85, edgecolor="white")
    for bar, val in zip(bars, top["arrests_per_100k"]):
        ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
                f"{val:.0f}", va="center", fontsize=8.5, color="#2c3e50")
    legend_handles = [
        mpatches.Patch(color=RED,  label="287(g) / no sanctuary"),
        mpatches.Patch(color=BLUE, label="Sanctuary / no 287(g)"),
        mpatches.Patch(color=GOLD, label="Mixed / neither"),
    ]
    ax.legend(handles=legend_handles, fontsize=9, loc="lower right", framealpha=0.9)
    ax.set_xlabel("ICE Arrests per 100,000 residents", fontsize=11)
    ax.set_title("Top 15 States by ICE Arrest Rate\n"
                 "Enforcement policy posture as primary differentiator",
                 fontsize=12, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", alpha=0.25, color=GREY)
    fig.tight_layout()
    fig.savefig(FIGURES_P1 / "fig_p1_04_top_states_arrests.png", dpi=180, bbox_inches="tight")
    plt.close()
    print("  Saved fig_p1_04")


# Fig P1-05: Model Coefficients
def fig_model_coefficients():
    coef_path = RESULTS_P1 / "model_coefficients_all.csv"
    if not coef_path.exists():
        print("  Skipping fig_p1_05 — run 05_model_enforcement.py first")
        return
    coefs = pd.read_csv(coef_path)
    TERM_LABELS = {
        "aai":                  "AAI (FQHC density)",
        "aai_x_foreign_born":   "AAI × Foreign-born share",
        "foreign_born_share":   "Foreign-born share",
        "poverty_rate":         "Poverty rate",
        "uninsured_share_proxy":"Uninsured share",
        "state_287g":           "287(g) agreement ★",
        "border_state":         "Border state",
        "sanctuary_policy":     "Sanctuary policy",
    }
    focus = coefs[coefs["model"] == "OLS_HC3_with_interaction"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 6), facecolor=BG)
    fig.suptitle("OLS Regression Coefficients: What Predicts ICE Enforcement Intensity?\n"
                 "(HC3 robust standard errors, N=51 states)",
                 fontsize=12, fontweight="bold")
    for i, outcome in enumerate(["arrests_per_100k", "detainers_per_100k"]):
        ax = axes[i]
        ax.set_facecolor(BG)
        sub = focus[(focus["outcome"] == outcome) & (focus["term"].isin(TERM_LABELS))].copy()
        sub["label"] = sub["term"].map(TERM_LABELS)
        sub = sub.sort_values("coef")
        bar_colors = [RED if c > 0 else BLUE for c in sub["coef"]]
        ax.barh(sub["label"], sub["coef"], color=bar_colors, alpha=0.82, edgecolor="white")
        ax.axvline(0, color="#1a1a1a", linewidth=1)
        for _, row in sub.iterrows():
            stars = ("***" if row["p_value"] < 0.01 else
                     "**"  if row["p_value"] < 0.05 else
                     "*"   if row["p_value"] < 0.10 else "n.s.")
            color = RED if stars != "n.s." else GREY
            offset = 5 if row["coef"] > 0 else -5
            ha = "left" if row["coef"] > 0 else "right"
            ax.annotate(stars, (row["coef"], row["label"]),
                        xytext=(offset, 0), textcoords="offset points",
                        ha=ha, va="center", fontsize=9,
                        color=color, fontweight="bold")
        title_str = "Arrests / 100k" if i == 0 else "Detainers / 100k"
        ax.set_title(title_str, fontsize=11, fontweight="bold")
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="x", alpha=0.2, color=GREY)
    fig.tight_layout()
    fig.savefig(FIGURES_P1 / "fig_p1_05_model_coefficients.png", dpi=180, bbox_inches="tight")
    plt.close()
    print("  Saved fig_p1_05")


# Fig P1-06: AAI Distribution
def fig_aai_distribution(df):
    fig, ax = plt.subplots(figsize=(8, 5), facecolor=BG)
    ax.set_facecolor(BG)
    ax.hist(df[df["state_287g"] == 0]["aai"], bins=12,
            color=BLUE, alpha=0.65, label="No 287(g)", edgecolor="white")
    ax.hist(df[df["state_287g"] == 1]["aai"], bins=12,
            color=RED, alpha=0.65, label="287(g) state", edgecolor="white")
    ax.set_xlabel("Administrative Accessibility Index (z-score)", fontsize=11)
    ax.set_ylabel("Number of States", fontsize=11)
    ax.set_title("AAI Distribution by 287(g) Status\n"
                 "287(g) states are not systematically higher in FQHC density",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIGURES_P1 / "fig_p1_06_aai_distribution.png", dpi=180, bbox_inches="tight")
    plt.close()
    print("  Saved fig_p1_06")


def main():
    print("Generating Phase 1 figures...")
    df = load()
    fig_aai_vs_arrests(df)
    fig_aai_vs_detainers(df)
    fig_287g_gap(df)
    fig_top_states(df)
    fig_model_coefficients()
    fig_aai_distribution(df)
    print(f"\nAll Phase 1 figures saved to {FIGURES_P1}")


if __name__ == "__main__":
    main()
